# Phase 1: Comprehensive Data Audit & Quality Profiling
**SAS CU Hackathon Analytics Project — Round 2**  
*Lead Data Scientist & Analytics Architecture Team*  

This notebook performs an exhaustive, reproducible audit of the **FOUR RAW DATASETS**.
In strict compliance with Phase 1 non-negotiable rules, **NO RAW DATA IS MODIFIED, IMPUTED, OR CLEANED** in this notebook.

In [1]:
import os
import sys
import hashlib
import pandas as pd
import numpy as np

# Ensure project root is in sys.path
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.data.loaders import load_csv_dataset, load_excel_dataset, load_all_raw_datasets
from src.data.profiling import (
    profile_dataframe, profile_missing_values, profile_duplicates,
    profile_identifier, profile_numeric_columns, detect_potential_outliers,
    profile_categorical_columns, profile_target_distribution
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
print("Environment initialized successfully. Pandas version:", pd.__version__)

## 1. Raw Data Verification & File Hash Integrity
Verifying that raw datasets exist and match byte-for-byte with preserved copies in .

In [2]:
raw_dir = os.path.join(project_root, "data/raw")
files = ["DataScience Jobs.csv", "Analytics Jobs.csv", "JDS Skill Traits.xlsx", "SDS Personality Traits.xlsx"]

for f in files:
    root_p = os.path.join(project_root, f)
    raw_p = os.path.join(raw_dir, f)
    with open(root_p, "rb") as fp1, open(raw_p, "rb") as fp2:
        h1 = hashlib.sha256(fp1.read()).hexdigest()
        h2 = hashlib.sha256(fp2.read()).hexdigest()
    assert h1 == h2, f"Hash mismatch for {f}"
    print(f"[VERIFIED] {f}: {os.path.getsize(raw_p):,} bytes | SHA256: {h1[:16]}...")

## 2. Load Raw Datasets in Read-Only Mode
Loading all four datasets using modular loaders.

In [3]:
datasets = load_all_raw_datasets(raw_dir)
for name, df in datasets.items():
    prof = profile_dataframe(df, name)
    print(f"{name}: Shape {df.shape} | Memory: {prof['memory_usage_mb']} MB | Columns: {len(df.columns)}")

## 3. Data Type Profiling
Cataloging storage dtypes vs semantic types.

In [4]:
from src.data.audit import run_full_phase1_audit
out_dir = os.path.join(project_root, "outputs/tables")
audit_results = run_full_phase1_audit(raw_dir=raw_dir, output_dir=out_dir)
audit_results["data_type_profile"].head(10)

## 4. Missing Value Diagnostics
Evaluating actual NaN counts alongside string representations of missingness.

In [5]:
mv = audit_results["missing_value_profile"]
mv[mv["total_missing_count"] > 0]

## 5. Duplicate Rows & Identifier Uniqueness
Auditing exact duplicate rows and primary identifier multiplicity.

In [6]:
print("=== DUPLICATE PROFILE ===")
display(audit_results["duplicate_profile"])
print("
=== IDENTIFIER AUDIT ===")
display(audit_results["id_audit"])

## 6. Numerical Variables & Outliers
Computing robust five-number summaries and Tukey IQR outlier boundaries.

In [7]:
print("=== NUMERICAL VARIABLES ===")
display(audit_results["numerical_profile"])
print("
=== OUTLIER AUDIT (Tukey 1.5x IQR) ===")
display(audit_results["outlier_profile"])

## 7. Categorical & Text Profiling
Evaluating category frequencies, whitespace issues, and casing variability.

In [8]:
audit_results["categorical_profile"].head(10)

## 8. Target Distributions & Class Balance
Auditing JDS and SDS binary classification target balance.

In [9]:
audit_results["target_class_balance"]

## 9. Consolidated Anomaly Inventory
Summary of all structural anomalies documented for Phase 2 resolution.

In [10]:
audit_results["anomaly_inventory"]